In [1]:
import xgboost as xgb
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import geoplot as gplt
import geoplot.crs as gcrs
from sklearn.model_selection import train_test_split
import contextily as cx

In [2]:
data = pd.read_csv('../data/seattle_sample_3k.csv')

In [3]:
data = gpd.GeoDataFrame(data, crs="EPSG:32610", geometry=gpd.points_from_xy(x=data.UTM_X, y=data.UTM_Y))

In [4]:
data = data.to_crs(4326)

In [5]:
data['lat'] = data['geometry'].get_coordinates()['x']
data['lon'] = data['geometry'].get_coordinates()['y']

In [6]:
data['price'] = np.power(10, data['log_price']) / 10000

In [7]:
y = data.price
X = data[['bathrooms', 'sqft_living', 'sqft_lot', 'grade', 'condition', 'waterfront', 'view', 'age', 'UTM_X', 'UTM_Y']]
loc = data[['lat','lon']]

In [8]:
X_train, X_temp, y_train, y_temp, loc_train, loc_temp = train_test_split(X, y, loc, train_size=0.8, random_state=42)

In [9]:
X_calib, X_test, y_calib, y_test, loc_calib, loc_test = train_test_split(X_temp, y_temp, loc_temp, train_size=0.5, random_state=42)


In [10]:
model = xgb.XGBRegressor(n_estimators=500, max_depth=3, min_child_weight=1.0, colsample_bytree=1.0)

In [11]:
model.fit(X_train, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=1.0, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             gamma=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=None, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=3, max_leaves=None,
             min_child_weight=1.0, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=500, n_jobs=None,
             num_parallel_tree=None, random_state=None, ...)

In [12]:
model.score(X_test, y_test)

0.8705986887146308

In [13]:
from GeoConformal import GWQRBasedGeoConformalSpatialPrediction

In [14]:
geocp = GWQRBasedGeoConformalSpatialPrediction(predict_f=model.predict, nonconformity_score_f=None, miscoverage_level=0.1, alpha=8, coord_calib=loc_calib.values, coord_test=loc_test.values, x_calib=X_calib, y_calib=y_calib, x_test=X_test, y_test=y_test)

In [15]:
results = geocp.analyze()

100%|██████████| 300/300 [02:34<00:00,  1.94it/s]


In [16]:
results.coverage_probability

0.91

In [17]:
results.geo_uncertainty.mean() / 2

12.519958455186265

In [18]:
from GeoConformal import GeoConformalSpatialPrediction

In [19]:
geocp_spatial_std = GeoConformalSpatialPrediction(predict_f=model.predict, nonconformity_score_f=None, miscoverage_level=0.1, bandwidth=0.15, coord_calib=loc_calib.values, coord_test=loc_test.values, X_calib=X_calib, y_calib=y_calib, X_test=X_test, y_test=y_test)

In [20]:
results_std = geocp_spatial_std.analyze()

In [21]:
results_std.coverage_probability

0.9366666666666666

In [22]:
results_std.geo_uncertainty.mean()

19.452105716929108